# Data processing: Bakery sales


## About the dataset

The dataset belongs to a French bakery. The dataset provides the daily transaction details of customers from 2021-01-01 to 2022-09-30.
Yearly and weekly saisonalities can be observed.

**Content**  
The dataset has 234005 entries, over 136000 transactions and 6 columns.

**Variables**  
* ``date``: date of the order
* ``time``: time of the order
* ``ticket_number``: identifier for every single transaction; 1 ticket_number = is a single order
* ``article``: name of the product sold (in French)
* ``quantity``: quantity sold
* ``unit_price``: price per product
* `retour`: whether a data entry is part of a retour
    * *False*: products sold, not retoured
    * *True*: This is a retour
    * *product_number*: these products are retoured, the product_number of the retour is referenced
    * *partial*: these products are partially retoured

**Note**
* The product COUPE is when customers ask to slice (with a machine) their whole bread.

## Assignment

Import the necessary packages

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

### Data cleaning and preprocessing

* Load the data in a dataframe named `df_bakery`
    * Use the first column as indices
* Show the first 6 rows to inspect the dataframe (do not use slicing!)

In [ ]:
df_bakery = pd.read_csv("bakery_sales.csv", index_col= 0)
df_bakery.head(6)

As you notice the indices miss certain numbers, let's fix that. 
* The new index should start from 0 and include all integers (0,1,2,3 etc.)
* The original index should not be added as a column to the dataframe


In [ ]:
df_bakery = df_bakery.reset_index(drop=True)

#Alternative b:
#df_bakery.index = [i for i in range(len(df_bakery))]

* Rename the column "Quantity" to `quantity` and the column "unit_price" to `unit_price_euro`
    * Do this without changing the order of the columns
    * Do this in one line of code


In [ ]:
df_bakery= df_bakery.rename(columns={"Quantity":"quantity", "unit_price": "unit_price_euro"})
df_bakery


* Remove rows where ``quantity`` is not provided

In [ ]:
df_bakery = df_bakery.dropna(subset=["quantity"])

* Change the dates in the column `date` to pandas datetime objects.

In [ ]:
df_bakery["date"] = pd.to_datetime(df_bakery["date"], format="%Y*%m*%d")


In [ ]:
df_bakery.info()

* Add a column `day` containing the name of the day of the week (Monday, Tuesday etc.)

In [ ]:
df_bakery["day"] = df_bakery.loc[:,"date"].dt.day_name()
# df_bakery["day"] = df_bakery["date"].dt.day_name()

* Convert the `quantity` and `ticket_number` columns to integers.

In [ ]:
df_bakery["quantity"] = df_bakery["quantity"].apply(int)
df_bakery["ticket_number"] = df_bakery["ticket_number"].astype(int)

#### Subtotal
* Add a column `subtotal` containing the total price of the sold item(s) per row
* In order to achieve this, you will have to convert the `unit_price_euro` to a float. Do this by writing your own function.
    * You can assume all prices contain a numerical value and are in euro, so you do not have to convert the values.
    * However, there might be inconsistencies in the data, ensure your function is robust enough to correctly handle this messy data without throwing an error.

In [ ]:
#Unit_price_euro
def unit_price(row):
    up = str(row["unit_price_euro"])
    up = up.split(" ")[0].strip()
    up = up.replace(",", ".") #up = moet ervoor, doet niet in place
    return float(up)

df_bakery["unit_price_euro"] = df_bakery.apply(unit_price, axis=1)

In [ ]:
#Subtotal
df_bakery["subtotal"] = df_bakery["quantity"]*df_bakery["unit_price_euro"]
df_bakery

## Statistics

* Display the ticketnumber of the most expensive **order**, which is not (partially) returned.

In [ ]:
df_bakery[df_bakery["retour"]=="False"].groupby("ticket_number")["subtotal"].sum().idxmax()

#### Closing day

As you might have noticed, this bakery is often opened every day of the week. The owner wants to introduce a closing day and asks for your help.
1. First you take a look at the **total income** for **every weekday**.
* Calculate and display the total income made on each weekday (Monday, Tuesday etc.)
* Display the day of the week and its total income for the day with the least income


In [ ]:
total_income = df_bakery.groupby("day")["subtotal"].sum().sort_values(ascending=True)
print(total_income)
total_income.iloc[[0]]
#head ook goed rekenen

2. Secondly you take a look at the **average income** for **every weekday**
* Calculate and display the average income made on each weekday (Monday, Tuesday etc.)
* Display the day of the week for the day with the smallest average income.

In [ ]:
grouped = df_bakery.groupby(["day","date"])["subtotal"].sum().groupby("day")
mean_income = grouped.mean()
print(mean_income)
print(mean_income.idxmin())

In [ ]:
# er is een makkelijkere (maar niet mooie) workaround voor studenten die geen dubbele group by kunnen. Wel geen full points.
data = df_bakery["date"].unique()
days = data.day_name()
dagen = pd.Series(days).value_counts()
print(dagen)
for idx in dagen.index:
    print(idx, total_income.loc[idx]/dagen.loc[idx])

3. Briefly explain why the day with the smallest average income and the day with the smallest total income are not the same. Display a statistic to clarify your answer.

In [ ]:
# Some Wednesdays the bakery was already closed.
grouped.count()

#### Categorizing orders by time of day

*  Define a function that classifies each transaction based on the **`time`** of the order.
* Use the following categories:

   * Before 10:00 → `morning`
   * From 10:00 to 13:00 (13:00 not included) → `late morning - noon`
   * From 13:00 to 16:00 (16:00 not included) → `noon - afternoon`
   * From 16:00 to 19:00 (19:00 not included) → `early evening`
   * As from 19:00 → `evening`
* Apply the function to create a new column called `time_slot`.

In [ ]:
df_bakery["time"] = pd.to_datetime(df_bakery["time"], format="%H:%M").dt.time #voor code hieronder

In [ ]:
def when(row):
    if row["time"] < pd.to_datetime("10:00",format="%H:%M").time():
        return "morning"
    elif row["time"] < pd.to_datetime("13:00",format="%H:%M").time():
        return "late morning - noon"
    elif row["time"] < pd.to_datetime("16:00",format="%H:%M").time():
        return "noon - afternoon"
    elif row["time"] < pd.to_datetime("19:00",format="%H:%M").time():
        return "early evening"
    else:
        return "Evening"
# werkt ook als de pure string wordt vergeleken, is ook ok. Full points.

In [ ]:
df_bakery["time_slot"] = df_bakery.apply(when, axis=1)

* Display the different time_slot's with the number of **orders** placed during that time slot.
    * Display the time slots with the least **orders** first, and with the most last.

In [ ]:
df_bakery.groupby("time_slot")["ticket_number"].count().sort_values(ascending=True)

## Graph

* Create a new dataframe `df_bakery_summer` only containing the data for the months June, July and August 2021.

In [ ]:
df_bakery_summer = df_bakery[df_bakery["date"] < pd.to_datetime("2021-9-01")]
df_bakery_summer = df_bakery_summer[df_bakery_summer["date"] >= pd.to_datetime("2021-06-01")]

*If you did not manage to complete previous exercises, the resulting data is provided in the file `graph_data_summer.csv` which you can use to create the graph, instead of your own `df_bakery_summer` dataframe. The resulting graph is shown in the image "bakery_plot.png".*  

Plot a multi-line chart displaying the daily quantity sold of selected bakery items during the summer months (June, July, August) of 2021.

* The x-axis displays the date, in chronological order
* The y-axis displays the number of items sold
* Include the following four products:
  * Croissant: plotted in blue, solid line with circular markers of size 2
  * Pain au chocolat: plotted in brown, solid line with circular markers of size 2
  * Baguette: plotted in orange, dashed line, without any markers
  * Tarte fruits 6p: plotted in green, solid line with with circular markers of size 2
* All lines should have a line width of size 1
* The figure size should be 11 x 5
* Provide meaningful axis labels and a descriptive title
* Display a legend
* Display horizontal grid lines for better readability
* Provide x-axis ticks for every first and 15th day of the month, formatted as "01 August 2021"


In [ ]:
#PAIN AU CHOCOLAT
pac = df_bakery_summer[df_bakery_summer["article"]== "PAIN AU CHOCOLAT"].groupby("date")["quantity"].sum()
#CROISSANT
crois = df_bakery_summer[df_bakery_summer["article"]== "CROISSANT"].groupby("date")["quantity"].sum()
#BAGUETTE
bag = df_bakery_summer[df_bakery_summer["article"]== "BAGUETTE"].groupby("date")["quantity"].sum()
#TARTE FRUITS 6P
tf = df_bakery_summer[df_bakery_summer["article"]== "TARTE FRUITS 6P"].groupby("date")["quantity"].sum()

In [ ]:
plt.figure(figsize=(11,5))
plt.plot(crois.index, crois.values, color = "blue", linewidth = 1, marker="o", markersize = 2, label="Croissant")
plt.plot(pac.index, pac.values, color = "brown", marker="o", markersize = 2, linewidth = 1, label="Pain au chocolat")
plt.plot(bag.index, bag.values, color = "orange", linewidth = 1, label="Baguette", linestyle="--")
plt.plot(tf.index, tf.values, color = "green", marker = ".", label="Tarte fruits 6p")
plt.legend()
plt.grid(axis="y")
plt.ylabel("quantity")
plt.xlabel("date")
plt.title("Daily bakery sales data 2021")
locs, labs = plt.xticks()
plt.xticks(locs, [pd.to_datetime(i.get_text()).strftime("%d %B %Y") for i in labs])

plt.show()
#xlabs mag ook hardcoded